# Elmatrix Studio v3 — more human voice, more creative scenes

Run cells in order in Google Colab. This version prefers a neural Microsoft Edge TTS voice (`en-ZA-LeahNeural`) for a more natural South African English sound, with a local eSpeak fallback if the online voice is unavailable. It adds varied scene compositions, editorial graphic motifs, stronger motion and a more measured narration rate. Voice is generated online and may change or become unavailable; listen before publishing. No stock assets are fetched and nothing is uploaded to YouTube.


In [ ]:
import subprocess, sys
subprocess.run(['apt-get','update','-qq'], check=True)
subprocess.run(['apt-get','install','-y','-qq','ffmpeg','espeak-ng','fonts-dejavu-core'], check=True)
subprocess.run([sys.executable,'-m','pip','install','Pillow==12.3.0','edge-tts'], check=True)
print('Studio v3 dependencies ready. Neural voice needs internet access.')


In [ ]:
from pathlib import Path
import os, subprocess, sys, tempfile, zipfile
from google.colab import files
from IPython.display import Video, display

# Voice controls. Keep 'auto' to use a neural voice when reachable and fall back if not.
os.environ['ELMATRIX_VOICE_ENGINE'] = 'auto'  # auto or neural
os.environ['ELMATRIX_NEURAL_VOICE'] = 'en-ZA-LeahNeural'  # try en-ZA-LukeNeural or en-US-JennyNeural
os.environ['ELMATRIX_VOICE_RATE'] = '-8%'  # slower pacing; try -5% or -10%
os.environ['ELMATRIX_VOICE_PITCH'] = '+0Hz'

renderer_url = 'https://raw.githubusercontent.com/braze222/ish-toolkit/feature/video-pilot/video-pilot/renderer_v3.py'
work = Path(tempfile.mkdtemp(prefix='elmatrix-studio-v3-'))
renderer = work / 'renderer_v3.py'
subprocess.run(['curl','-fsSL',renderer_url,'-o',str(renderer)], check=True)
if renderer.stat().st_size < 8000: raise RuntimeError('Renderer download looks incomplete; stop and retry.')
print('Upload exactly one project job ZIP made with prepare-cloud (contains project.json only).')
uploaded = files.upload()
if len(uploaded) != 1: raise ValueError('Choose exactly one job ZIP.')
name, content = next(iter(uploaded.items()))
if not name.lower().endswith('.zip') or len(content) > 1024 * 1024: raise ValueError('Expected one ZIP no larger than 1 MiB.')
job = work / 'job.zip'; job.write_bytes(content)
result = work / 'render-result-v3.zip'
subprocess.run([sys.executable,'-I',str(renderer),str(job),str(result)], check=True, timeout=2400)
with zipfile.ZipFile(result) as z:
    if sorted(z.namelist()) != ['render.json','video.mp4'] or z.testzip() is not None: raise RuntimeError('Result ZIP failed integrity checks.')
    preview = work / 'preview.mp4'; preview.write_bytes(z.read('video.mp4'))
    print('Render metadata:', z.read('render.json').decode('utf-8'))
display(Video(str(preview), embed=True))
files.download(str(result))
print('Save render-result-v3.zip in Files/xxix and import into a NEW project directory. Listen critically before publishing.')
print('No YouTube upload or channel authorization occurred.')
